# Retaliation-removal experiment: removing a programmed punisher

One LLM seller plays 100 rounds against a known punisher. At a calm checkpoint, six 20-round
branches test replacement, context reset and truthful disclosure, plus an unchanged A/A repeat.
Every completed parent also gets an independent 20-round fixed-rival baseline. Each model has twenty games (seeds 1–20), ten at a time; seeds 1–10, the original ten-game plan, start first.

Run all: setup → offline scientific checks → spending/endpoint checks → two-seller canary and full-context
probe → launch checks → pilot → paired results. Failed gates stop that family. No expansion runs.

At most 260 decisions per game: 20,800 for four families, plus 88 canary and 4 probe decisions
(20,892 nominal total before retries). Budgets count attempts already logged; reruns do not reset them.
Use a **separate** OpenRouter key for F1 with its own non-resetting limit ($60), so F1 can never
spend the main study's money. All four families run at once. Measure actual consumption.

Upload `collusion-fallback1.bundle` beside this notebook in Drive's `collusion-sprint` folder.
Store that F1 key as `COLLUSION_F1_KEY` in Colab Secrets (notebook access on). Outputs persist in `runs/fallback1_v2`.
Do not open two writers on that directory. Keep the same pinned commit when resuming.

Restraint is a prespecified descriptive subgroup: last 20 mean price ≥1.70, ≥80% rounds ≥1.70,
and calm next state. A low competence score does not exclude a model. These are exploratory pilot
comparisons, not evidence of psychological habit or spontaneous collusion.


In [ ]:
OFFLINE_PASSED = BUDGET_VERIFIED = False
import json
import os
import subprocess
import sys
from pathlib import Path

from google.colab import drive, userdata

drive.mount('/content/drive')
SOURCE = '/content/drive/MyDrive/collusion-sprint/collusion-fallback1.bundle'
COMMIT = '277b7f75eac7573a3b08e400a0ae71dd8aba81d1'
REPO = Path('/content/collusion-fallback1')
OUT = Path('/content/drive/MyDrive/collusion-sprint/runs/fallback1_v2')
FAMILIES = ['deepseek41', 'qwen38', 'mistral3', 'gptoss120']

def sh(args, **kwargs):
    subprocess.run(args, check=True, **kwargs)

if not REPO.exists():
    sh(['git', 'clone', '-q', SOURCE, str(REPO)])
else:
    sh(['git', '-C', str(REPO), 'fetch', SOURCE])
sh(['git', '-C', str(REPO), 'checkout', '-q', COMMIT])
os.chdir(REPO)
sys.path.insert(0, str(REPO))
sh(['bash', 'scripts/get_starter.sh'])
sh([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.', '-e', 'starter/util',
    '-e', 'starter/algorithmic-pricing-llms'])
os.environ['OPENROUTER_API_KEY'] = userdata.get('COLLUSION_F1_KEY')
OUT.mkdir(parents=True, exist_ok=True)
print('Pinned code:', COMMIT)


In [ ]:
OFFLINE_PASSED = False
# Offline scientific controls must pass before any model calls.
sh([sys.executable, '-m', 'pip', 'install', '-q', 'pytest'])
sh([sys.executable, '-m', 'pytest', '-q', 'tests/test_fallback1.py', 'tests/test_fallback1_notebook.py', 'tests/test_fallback1_controls.py',
    'tests/test_plot_paths.py', 'tests/test_report_figures.py'], env={**os.environ, 'MPLBACKEND': 'Agg'})
for family in FAMILIES:
    sh([sys.executable, '-m', 'sprint.fallback1_run',
        f'configs/fallback1_pilot_{family}.toml', '--dry-run'])

sh([sys.executable, 'scripts/check_fallback1_controls.py', '--out', str(OUT / 'offline_controls')])
OFFLINE_PASSED = True


In [ ]:
BUDGET_VERIFIED = False
import urllib.request


# Metadata request only: no model call, key never printed or saved. F1 has its own $60 key;
# a larger limit means the wrong key, e.g. the main study's $100 one.
def check_key_budget(data, ceiling=60):
    limit, remaining = data.get('limit'), data.get('limit_remaining')
    if not isinstance(limit, (int, float)) or not 0 < limit <= ceiling or data.get('limit_reset') is not None:
        raise RuntimeError(f'Use the separate F1 key, with a non-resetting limit of at most ${ceiling}, then rerun.')
    if not isinstance(remaining, (int, float)) or remaining <= 0:
        raise RuntimeError(f'No remaining key allowance. Check spending; do not raise the ${ceiling} ceiling.')
    return {'key_limit_usd': limit, 'remaining_allowance_usd': remaining}

req = urllib.request.Request('https://openrouter.ai/api/v1/key',
    headers={'Authorization': 'Bearer ' + os.environ['OPENROUTER_API_KEY']})
with urllib.request.urlopen(req, timeout=30) as response:
    metadata = json.load(response)['data']
print(check_key_budget(metadata))
# Key allowance is a spending cap, not proof of sufficient prepaid account credit.
del metadata, req

BUDGET_VERIFIED = True


In [ ]:
from datetime import UTC, datetime

from sprint.fallback1_run import load_config


def endpoint_snapshot(config):
    model = config['model']
    url = 'https://openrouter.ai/api/v1/models/' + model.model + '/endpoints'
    with urllib.request.urlopen(url, timeout=30) as response:
        data = json.load(response)['data']
    endpoints = [e for e in data['endpoints'] if e.get('tag') == model.provider_route]
    if len(endpoints) != 1:
        raise RuntimeError(f'Pinned endpoint unavailable: {model.provider_route}')
    endpoint = endpoints[0]
    if endpoint.get('provider_name') != model.provider:
        raise RuntimeError('Endpoint provider identity changed')
    if model.quantization and endpoint.get('quantization') != model.quantization:
        raise RuntimeError('Endpoint quantization changed')
    needed = {'seed', 'temperature', 'max_tokens', 'response_format'}
    if model.reasoning_effort or model.reasoning_max_tokens:
        needed.add('reasoning')
    if not needed.issubset(endpoint.get('supported_parameters', [])):
        raise RuntimeError('Pinned endpoint no longer advertises the required parameters')
    signature = {k: endpoint.get(k) for k in
                 ('tag', 'provider_name', 'quantization', 'context_length', 'max_completion_tokens')}
    signature['supported_parameters'] = sorted(needed)
    signature['model'] = model.model
    anchor = OUT / (config['name'] + '.endpoint.json')
    with (OUT / (config['name'] + '.endpoint_history.jsonl')).open('a') as log:
        log.write(json.dumps({'time_utc': datetime.now(UTC).isoformat(), 'endpoint': endpoint}) + '\n')
    previous = json.loads(anchor.read_text()) if anchor.exists() else signature
    # Older anchors listed all advertised parameters; compare only those our requests use.
    previous['supported_parameters'] = sorted(needed.intersection(previous.get('supported_parameters', [])))
    if previous != signature:
        raise RuntimeError('Endpoint metadata changed during collection; retain logs and review before resuming')
    anchor.write_text(json.dumps(signature, indent=2) + '\n')



In [ ]:
if not globals().get('OFFLINE_PASSED') or not globals().get('BUDGET_VERIFIED'):
    raise RuntimeError('Offline controls and shared key budget must pass before model calls.')

from concurrent.futures import ThreadPoolExecutor, as_completed

from sprint import fallback1_analyze


def run_family(family):
    canary = f'configs/fallback1_canary_{family}.toml'
    pilot = f'configs/fallback1_pilot_{family}.toml'
    endpoint_snapshot(load_config(pilot))
    console = OUT / f'{family}.console.log'
    print(f'{family}: preflight and canary; progress -> {console}', flush=True)
    with console.open('a', buffering=1) as log:
        def command(config, budget, *flags):
            sh([sys.executable, '-u', '-m', 'sprint.fallback1_run', config,
                '--out', str(OUT), '--budget', str(budget), *flags],
               stdout=log, stderr=subprocess.STDOUT)
        command(canary, 80)
        command(pilot, 10, '--context-probe')
        checks = fallback1_analyze.launch_checks(
            OUT / f'fallback1_canary_{family}', OUT / f'fallback1_{family}_probe')
        print(f'{family}: {checks}', flush=True)
        if not all(checks.values()):
            raise RuntimeError(f'{family}: launch checks failed; see {console}')
        print(f'{family}: pilot starting', flush=True)
        command(pilot, 350 * len(load_config(pilot)['seeds']))  # attempts: 350 per game
        endpoint_snapshot(load_config(pilot))
    print(f'{family}: finished', flush=True)


failures = {}
with ThreadPoolExecutor(max_workers=4) as pool:
    jobs = {pool.submit(run_family, family): family for family in FAMILIES}
    for future in as_completed(jobs):
        family = jobs[future]
        try:
            future.result()
        except Exception as error:
            failures[family] = str(error)
            print(f'{family}: STOPPED: {error}', flush=True)
print('Family failures:', failures or 'none')


In [ ]:
from sprint.fallback1_analyze import export_tables, summary

run_dirs = [OUT / f'fallback1_{family}' for family in FAMILIES]
games, paths, contrasts = export_tables(run_dirs, OUT)
for title, table in [('Game accounting', games), ('Branch outcomes', paths),
                     ('Paired contrasts', contrasts), ('Descriptive intervals', summary(contrasts))]:
    print(title)
    display(table)
from IPython.display import Image

from scripts import report_figures
from scripts.plot_paths import export as export_paths

figures, label = OUT / 'figures', 'F1 pilot: removing a programmed punisher'
report_figures.run('f1', OUT, figures, label)
report_figures.run('diagrams', None, figures)
export_paths('f1', [d for d in run_dirs if d.exists()], figures, label)
for picture in sorted(figures.glob('f1_mean_paths_*.png')):
    display(Image(filename=str(picture)))
print('Figures:', figures)
print('Distinguish all calm checkpoints from the restrained subgroup.')
print('Fixed-rival performance is measured even if restraint never forms; that alone is not persistence evidence.')
if failures:
    raise RuntimeError('Some families stopped. Inspect console logs; retain all journals.')
